# Live exercise: session 7, in pandas, worked solution

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")
plays = pd.read_sql("SELECT * FROM plays", con)
artists = pd.read_sql("SELECT * FROM artists", con)
awards = pd.read_sql("SELECT * FROM awards", con)
con.close()

print(plays.shape, artists.shape, awards.shape)

# Part 1

## 1. Long plays on the phone or the laptop

In [ ]:
long_play = plays["minutes_played"] > 5
phone_or_laptop = (plays["device"] == "phone") | (plays["device"] == "laptop")

with_brackets = long_play & phone_or_laptop
with_isin = long_play & plays["device"].isin(["phone", "laptop"])

assert with_brackets.sum() == with_isin.sum() == 523
print(with_brackets.sum(), "long plays on the phone or the laptop")

Giving each condition a name first, as here, is the easiest way to keep the
brackets straight. It reads like the sentence it came from.

The version people actually write first, all on one line, is where it goes
wrong:

In [ ]:
one_line_wrong = (plays["minutes_played"] > 5) & (plays["device"] == "phone") | (plays["device"] == "laptop")
print(one_line_wrong.sum())

871, with no error. `&` is worked out before `|`, so this means "long phone
plays, or **any** laptop play". The `or` part needs its own brackets around
it, exactly as `OR` does in SQL.

## 2. Total minutes per country

In [ ]:
named = plays.merge(artists, on="artist_id")
print(len(plays), "->", len(named), "rows")

per_country = (
    named.groupby("country")
    .agg(plays=("play_id", "count"), minutes=("minutes_played", "sum"))
    .sort_values("minutes", ascending=False)
)
per_country.head().round(1)

In [ ]:
assert len(named) == len(plays)            # many plays to one artist: no change
assert len(per_country) == 25
assert round(per_country.loc["Germany", "minutes"], 1) == 1285.1
print(len(per_country), "countries")

Germany 1285.1, Netherlands 1265.2, Norway 1184.0: the same as session 7.

`.loc["Germany", "minutes"]` reads one value by its labels: the row called
Germany, the column called minutes. The country is the index after a
`groupby`, which is what makes that possible.

## 3. The top three artists in 2025, by minutes

In [ ]:
in_2025 = named[named["played_at"] >= "2025-01-01"]

top_2025 = (
    in_2025.groupby("artist_name")
    .agg(plays=("play_id", "count"), minutes=("minutes_played", "sum"))
    .sort_values("minutes", ascending=False)
    .head(3)
)
top_2025.round(1)

In [ ]:
assert list(top_2025.index) == ["DJ Kompas", "Glass Tram", "Fjord & Flint"]
assert round(top_2025.loc["DJ Kompas", "minutes"], 1) == 570.9
assert top_2025.loc["Glass Tram", "plays"] == 115
assert top_2025.loc["DJ Kompas", "plays"] == 103

The filter comes first, on the rows, because it has nothing to do with the
grouping: that is the `WHERE` from session 7. Filter rows as early as you
can; every line after it has less to do.

Look at the plays column. Glass Tram has 115 plays in 2025 against DJ
Kompas's 103, yet DJ Kompas has the most minutes. (115 is also the tie with
Sara Lindqvist from your session 4 homework.) Most plays and most minutes
are different questions, as they were in sessions 4 and 7.

# Part 2

## 4. Merge awards on as well

In [ ]:
with_awards = named.merge(awards, on="artist_id")
print(len(named), "->", len(with_awards), "rows")

broken = (
    with_awards.groupby("country")
    .agg(rows_now=("play_id", "count"), minutes=("minutes_played", "sum"))
    .sort_values("minutes", ascending=False)
)
broken.head().round(1)

In [ ]:
assert len(with_awards) == 2371
assert len(broken) == 14
assert round(broken.loc["Norway", "minutes"], 1) == 2538.5
print(len(broken), "countries left")

## 5. What happened

* before: 2,183 rows, 25 countries, Germany 1285.1, Netherlands 1265.2,
  Norway 1184.0
* after: 2,371 rows, 14 countries, Norway 2538.5, Germany 2350.1,
  Ireland 1318.0

## 6. Why

The awards table has one row per award, and an artist can have several.
Fjord & Flint, from Norway, has 3 awards and 237 plays. The merge pairs every
play with every award for that artist, so 237 plays become 711 rows and
their minutes are counted three times. Norway's 2538.5 is Fjord & Flint's
846.2 three times over, give or take rounding, and nothing else: Norway's
other plays, from an artist with no awards, have dropped out entirely.

And the second bug, as in session 7: 25 countries became 14. `merge` is an
inner join unless you say otherwise, so every country whose artists have
never won an award vanished. The numbers left are inflated **and** rows are
missing.

Same cause, same symptoms, same name. pandas did nothing different from SQL.

## 7. The fix: aggregate first, then merge

In [ ]:
per_artist = (
    plays.groupby("artist_id")
    .agg(plays=("play_id", "count"), minutes=("minutes_played", "sum"))
    .reset_index()
)
awards_per_artist = awards.groupby("artist_id").size().rename("awards").reset_index()

fixed = (
    per_artist
    .merge(artists, on="artist_id")
    .merge(awards_per_artist, on="artist_id", how="left")
)
fixed["awards"] = fixed["awards"].fillna(0).astype(int)

fixed_country = (
    fixed.groupby("country")
    .agg(plays=("plays", "sum"), minutes=("minutes", "sum"), awards=("awards", "sum"))
    .sort_values("minutes", ascending=False)
)
fixed_country.head().round(1)

In [ ]:
assert len(fixed_country) == 25
assert list(fixed_country.index) == list(per_country.index)
assert (fixed_country["minutes"].round(1) == per_country["minutes"].round(1)).all()
print("all 25 countries back, every total matches query 2")

Two details worth pointing at:

* Both tables are reduced to **one row per artist** before anything is
  joined. Get both sides down to the same grain, then join. That is the
  general recipe, in any language.
* The awards merge is `how="left"`, so artists with no awards stay in, and
  `fillna(0)` turns their missing count into 0, which is what it means here.

The last `assert` compares two whole columns at once. `==` between two
Series gives a mask, and `.all()` asks whether every value in it is `True`.

## 8. Make pandas catch it

In [ ]:
try:
    named.merge(awards, on="artist_id", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

In [ ]:
checked = (
    per_artist
    .merge(artists, on="artist_id", validate="one_to_one")
    .merge(awards_per_artist, on="artist_id", how="left", validate="one_to_one")
)
print(len(checked), "rows, both checks passed")

The broken merge now refuses to run. The fixed one passes, because after
aggregating, every table really does have one row per artist.

`validate` would have caught this in session 7 too, if SQL had it. It does
not, which is one honest point in pandas' favour when a join is risky.

## If you have time: plays and average per genre

In [ ]:
per_genre = (
    named.groupby("genre")
    .agg(plays=("play_id", "count"), avg_min=("minutes_played", "mean"))
    .round(2)
)

by_plays = per_genre.sort_values("plays", ascending=False)
by_average = per_genre.sort_values("avg_min", ascending=False)

print(by_plays, "\n")
print(by_average)

In [ ]:
rank_plays = list(by_plays.index)
rank_average = list(by_average.index)

for genre in rank_plays:
    moved = rank_plays.index(genre) - rank_average.index(genre)
    print(f"  {genre:12} {moved:+d}")

assert rank_plays.index("Jazz") == 6 and rank_average.index("Jazz") == 0

Jazz moves furthest: seventh by plays, first by average, six places up.
Rock climbs five, and Pop drops five, from third to last. The ranking you show someone depends
entirely on which of those two questions you decided they were asking.